### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown


from openai import AsyncOpenAI
from agents import set_default_openai_client, set_default_openai_api

load_dotenv(override=True)

True

In [2]:
set_default_openai_client(
    AsyncOpenAI(
        base_url="http://localhost:1234/v1"
        # api_key="lm-studio"  # any non-empty string works
    )
)


set_default_openai_api("chat_completions")

In [3]:
from accounts import Account

In [4]:
account = Account.get("Ed")
account

Account(name='ed', balance=9657.315999999999, strategy='', holdings={'AMZN': 12}, transactions=[3 shares of AMZN at 31.062 each., 3 shares of AMZN at 42.084 each., 3 shares of AMZN at 27.054 each., 3 shares of AMZN at 14.028 each.], portfolio_value_time_series=[('2025-10-24 00:32:29', 10089.814), ('2025-10-24 00:32:32', 9978.814), ('2025-10-24 00:34:01', 10134.562), ('2025-10-24 00:34:03', 10350.562), ('2025-10-24 00:34:18', 9924.4), ('2025-10-24 00:34:18', 10392.4), ('2025-10-24 00:49:20', 10485.315999999999), ('2025-10-24 00:49:20', 9969.315999999999), ('2025-10-24 00:49:40', 10809.315999999999), ('2025-10-24 00:49:40', 9669.315999999999)])

In [5]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 9473.949999999999, "strategy": "", "holdings": {"AMZN": 15}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 31.062, "timestamp": "2025-10-24 00:32:29", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 42.084, "timestamp": "2025-10-24 00:34:01", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 27.054, "timestamp": "2025-10-24 00:34:18", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 14.028, "timestamp": "2025-10-24 00:49:20", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 61.122, "timestamp": "2025-10-24 00:50:59", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-10-24 00:32:29", 10089.814], ["2025-10-24 00:32:32", 9978.814], ["2025-1

In [6]:
account.report()

'{"name": "ed", "balance": 9473.949999999999, "strategy": "", "holdings": {"AMZN": 15}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 31.062, "timestamp": "2025-10-24 00:32:29", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 42.084, "timestamp": "2025-10-24 00:34:01", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 27.054, "timestamp": "2025-10-24 00:34:18", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 14.028, "timestamp": "2025-10-24 00:49:20", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 61.122, "timestamp": "2025-10-24 00:50:59", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-10-24 00:32:29", 10089.814], ["2025-10-24 00:32:32", 9978.814], ["2025-10-24 00:34:01", 10134.562], 

In [7]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 31.062,
  'timestamp': '2025-10-24 00:32:29',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 42.084,
  'timestamp': '2025-10-24 00:34:01',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 27.054,
  'timestamp': '2025-10-24 00:34:18',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 14.028,
  'timestamp': '2025-10-24 00:49:20',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 61.122,
  'timestamp': '2025-10-24 00:50:59',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [8]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [9]:
mcp_tools

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None),
 Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None),
 Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', '

In [10]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
# model = "gpt-4.1-mini"
model="openai/gpt-oss-20b"

In [11]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


**Account Overview – Ed**

- **Cash Balance:** $9,473.95  
- **Stock Holdings:**  
  - Amazon (AMZN): 15 shares

Let me know if you’d like to do anything else with your account!

### Now let's build our own MCP Client

In [12]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, annotations=None), Tool(name='get_holdings', description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, annotations=None), Tool(name='buy_shares', description="Buy shares of a stock.\n\n    Args:\n        name: The name of the account holder\n        symbol: The symbol of the stock\n        quantity: The quantity of shares to buy\n        rationale: The rationale for the purchase and fit with the account's strategy\n    ", inputSchema={'properties': {'name': {'title': 'Name', 'ty

In [13]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Hi Ed! Your current cash balance is **$9,473.95**. Let me know if you’d like to review your holdings or make any trades.

In [14]:
context = await read_accounts_resource("ed")
print(context)

{"name": "ed", "balance": 9473.949999999999, "strategy": "", "holdings": {"AMZN": 15}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 31.062, "timestamp": "2025-10-24 00:32:29", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 42.084, "timestamp": "2025-10-24 00:34:01", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 27.054, "timestamp": "2025-10-24 00:34:18", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 14.028, "timestamp": "2025-10-24 00:49:20", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 61.122, "timestamp": "2025-10-24 00:50:59", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-10-24 00:32:29", 10089.814], ["2025-10-24 00:32:32", 9978.814], ["2025-10-24 00:34:01", 10134.562], [

In [15]:
from accounts import Account
Account.get("ed").report()

'{"name": "ed", "balance": 9473.949999999999, "strategy": "", "holdings": {"AMZN": 15}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 31.062, "timestamp": "2025-10-24 00:32:29", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 42.084, "timestamp": "2025-10-24 00:34:01", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 27.054, "timestamp": "2025-10-24 00:34:18", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 14.028, "timestamp": "2025-10-24 00:49:20", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 61.122, "timestamp": "2025-10-24 00:50:59", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-10-24 00:32:29", 10089.814], ["2025-10-24 00:32:32", 9978.814], ["2025-10-24 00:34:01", 10134.562], 

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>

In [17]:
instructions = "You are able to manage a todo list for a client, and answer questions about the todo list."
request = "My name is Ed and my todo list is under the name Ed. What's on my todo list?"
model="openai/gpt-oss-20b"
params = {"command": "uv", "args": ["run", "todo_server.py"]}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="todo_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("todo_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Here’s what you currently have on your pending to‑do list:

| ID | Description   |
|----|---------------|
| 1  | Sample Task   |
| 3  | Third Task    |

Let me know if you'd like to add, complete, or remove any of these items!